In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
import os

from itertools import combinations
from matplotlib.colors import to_rgb

# imports from "common" folder
from common.data_processing import process_data

### 1. EDM calculation

In [ ]:
# Compute the relative Error matrix for each site
RF  = pd.read_csv("../data/SHAP_RF_df.csv", index_col=0)
BRT = pd.read_csv("../data/SHAP_BRT_df.csv", index_col=0)
MLP = pd.read_csv("../data/SHAP_MLP_df.csv", index_col=0)
GAM = pd.read_csv("../data/SHAP_GAM_df.csv", index_col=0)

# Organize into dictionary
models = ['GAM','RF', 'BRT', 'MLP']
shap_data = {
    'GAM': GAM.values,
    'RF': RF.values,
    'BRT': BRT.values,
    'MLP': MLP.values,
    
}

# Compute relative error matrices per site
site_relative_error_matrices = {}

for site_idx in range(2778):  # 0-based index
    matrix = pd.DataFrame(index=models, columns=models, dtype=float)

    for ref_model in models:
        for cmp_model in models:
            if ref_model == cmp_model:
                matrix.loc[ref_model, cmp_model] = np.nan  
            else:
                S_ref = shap_data[ref_model][site_idx]
                S_cmp = shap_data[cmp_model][site_idx]
                norm_ref = np.linalg.norm(S_ref)
                rel_error = np.linalg.norm(S_ref - S_cmp) / norm_ref if norm_ref != 0 else np.nan
                matrix.loc[ref_model, cmp_model] = round(rel_error, 3)


    matrix.index.name = f"Compared Model"
    site_relative_error_matrices[site_idx + 1] = matrix

for site_id in range(1, 2779): 
    print(f"\nRelative Error Matrix for Site {site_id}:")
    print(site_relative_error_matrices[site_id])


In [ ]:
# Compute the averageRelative error per site - accroding to the reference model
avg_error_per_site_ref = []

for site_id, matrix in site_relative_error_matrices.items():
    for ref_model in models:
        row = matrix.loc[ref_model]
        valid_errors = row.dropna()
        avg_error = valid_errors.mean()
       
        avg_error_per_site_ref.append({
            "Site": site_id,
            "Reference_Model": ref_model,
            "Average_Relative_Error": round(avg_error, 3)
        })

avg_error_ref_df = pd.DataFrame(avg_error_per_site_ref)

print(avg_error_ref_df)

In [ ]:
avg_error_ref_df['Rank'] = avg_error_ref_df.groupby('Reference_Model')['Average_Relative_Error'] \
                                           .rank(method='min', ascending=False)

avg_error_ref_ranked_df = avg_error_ref_df.sort_values(['Reference_Model', 'Rank'])

avg_error_ref_ranked_df


#### 1.1 Boxplot of EDM or SHAP discrepancy

In [ ]:
# Plot the boxplot of EDM (SHAP discrepancy)
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,

    "legend.fontsize": 12,
    "legend.title_fontsize": 14,
    "legend.frameon": False,
})


model_order = ['GAM','RF', 'BRT', 'MLP']
ref_palette = {
    'GAM': sns.color_palette("Purples", 4)[1],
    'RF':  sns.color_palette("Blues", 4)[1],
    'BRT': sns.color_palette("Greens", 4)[1],
    'MLP': sns.color_palette("Reds", 4)[1]
}

records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[avg_error_ref_ranked_df['Reference_Model'] == model].copy()
    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])


plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df,
    x='Reference_Model',
    y='Mean_over_SDofMean',
    hue='Reference_Model',
    order=model_order,
    palette=ref_palette,
    showfliers=False
)


jitter_strength = 0.15  
np.random.seed(42)  

for i, model in enumerate(model_order):
    model_df = df[df['Reference_Model'] == model]
    q3 = model_df['Mean_over_SDofMean'].quantile(0.75)
    q1 = model_df['Mean_over_SDofMean'].quantile(0.25)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = model_df[model_df['Mean_over_SDofMean'] > upper_bound]

   
    x_vals = i + np.random.uniform(-jitter_strength, jitter_strength, size=len(model_df))
    y_vals = model_df['Mean_over_SDofMean'].values
    sites = model_df['Site'].values

   
    # dot_color = ref_palette[model]
    base_color = to_rgb(ref_palette[model])
    dot_color = tuple([0.8 * c for c in base_color])  

    plt.scatter(x_vals, y_vals, color = dot_color, alpha=0.6, s=30, zorder=2)


    # for xi, yi, site in zip(x_vals, y_vals, sites):
    #     if yi > upper_bound:
    #         plt.text(
    #             xi + 0.04,
    #             yi,
    #             str(site),
    #             fontsize=9,
    #             color='red',
    #             rotation=45,
    #             ha='left',
    #             va='center',
    #             zorder=3
    #         )


plt.xlabel("Reference Model", fontsize=16,labelpad=10)
# plt.ylabel("Mean / SD(Mean)", fontsize=14)
# plt.ylabel(r"$\mu(i)\,/ \,\sigma_{\mu}$", fontsize=16, labelpad=10)
# plt.title("Training", fontsize=16)
# plt.ylabel(r"$\mu_{j_{1}} (i)/\sigma_{\mu_{j_{1}}}$", fontsize=16, labelpad=10)
plt.ylabel("SHAP discrepancy", fontsize=16, labelpad=10)
plt.tick_params(axis = 'x', labelsize =14)
plt.tick_params(axis = 'y', labelsize =14)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



In [ ]:
# dataset with Mean/SD 
records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[avg_error_ref_ranked_df['Reference_Model'] == model].copy()
    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])

# Save SHAP discrepancy 
df.to_csv("../data/SHAP_discrepancy_training_table.csv", index=False)

# Collect outliers per model
outlier_dict = {}

for model in model_order:
    model_df = df[df['Reference_Model'] == model]
    q3 = model_df['Mean_over_SDofMean'].quantile(0.75)
    q1 = model_df['Mean_over_SDofMean'].quantile(0.25)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    outliers = model_df.loc[model_df['Mean_over_SDofMean'] > upper_bound, 'Site'].tolist()
    outlier_dict[model] = outliers

# Save the indices of outliers for all models
outliers_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in outlier_dict.items()]))
print(outliers_df)
outliers_df.to_csv("../data/train_outliers_table.csv", index=False)


### 2. Least EDM (SHAP discrepancy) points

In [ ]:
datasets = {
    model: df[df["Reference_Model"] == model].sort_values(by="Site")
    for model in df["Reference_Model"].unique()
}

df_gam = datasets["GAM"]
df_rf  = datasets["RF"]
df_brt = datasets["BRT"]
df_mlp = datasets["MLP"]

# Save SHAP discrepancy by reference model
for model, dset in datasets.items():
    dset.to_csv(f"../data/{model}_ref_model_SHAP_discrepancy.csv", index=False)


In [ ]:
lowest5 = df.nsmallest(5, "Mean_over_SDofMean") # this gives 5 rows with the smallest values
print(lowest5)
lowest5.to_csv("waterfallplots_least_discrepancy/lowest5.csv", index=False)


In [ ]:
lowest_by_model = (df.loc[df.groupby("Reference_Model")["Mean_over_SDofMean"].idxmin()].sort_values("Mean_over_SDofMean")) #groupby(...).idxmin() finds row
print(lowest_by_model)
# lowest_by_model.to_csv("waterfallplots_least_discrepancy/lowest_by_model.csv", index=False)

In [ ]:
lowest5_by_model = (
    df.sort_values("Mean_over_SDofMean")
      .groupby("Reference_Model")
      .head(5)
)

print(lowest5_by_model)

In [ ]:
lowest5_by_model = (
    df.groupby("Reference_Model")
      .apply(lambda g: g.nsmallest(5, "Mean_over_SDofMean"))
      .reset_index(drop=True)
      .sort_values(["Reference_Model", "Mean_over_SDofMean"])
)

print(lowest5_by_model)

# lowest5_by_model.to_csv("waterfallplots_least_discrepancy/lowest5_by_model.csv", index=False)

In [ ]:
least_SHAP_dis_indices = [1245, 2536, 200, 2304, 663, 1116] # Corresponding indices for selected least SHAP discrepancy points

### 3. Outliers of EDM (SHAP discrepancy)

In [ ]:
# # Data processing from data_processing.py
# train, test, y_train, y_test, X_train_scaled, X_test_scaled = process_data()

In [ ]:
# Combined train and test vertically
# traintest = pd.concat([train, test])

In [ ]:
# def one_based_to_zero_based(series: pd.Series):
#     s = series.dropna().astype(int)
#     return (s - 1).tolist()   # 1-based -> 0-based iloc positions

# # Build per-model outlier tables
# outlier_views = {}

# for model in tbl.columns:
#     pos  = one_based_to_zero_based(outliers_df[model])   
#     idxs = X_train_scaled.index[pos]               

#     # Scaled rows (preserve order of 'pos')
#     subset = X_train_scaled.iloc[pos].copy()

#     # Metadata from trainvaltest (REEF + site + raw year)
#     meta = traintest.loc[idxs, ['REEF_NAME', 'SITE_NO', 'site_name', 'year','site_longitude','site_latitude']].copy()

#     # Stitch together: add identifying columns to the left
#     subset.insert(0, 'orig_index', idxs)
#     subset.insert(1, 'REEF_NAME', meta['REEF_NAME'].values)
#     subset.insert(2, 'SITE_NO',   meta['SITE_NO'].values)
#     subset.insert(3, 'site_name', meta['site_name'].values)
#     subset.insert(4, 'year_raw',  meta['year'].values)   # avoid clashing with scaled 'year' column
#     subset.insert(5, 'longitude_raw',  meta['site_longitude'].values)  
#     subset.insert(6, 'latitude_raw',  meta['site_latitude'].values) 

#     outlier_views[model] = subset.reset_index(drop=True)
#     subset.to_csv(f"../data/train_outliers_rows_{model}.csv", index=False)

# # Number of outlers
# for model, df_out in outlier_views.items():
#     print(f"{model}: {len(df_out)} outliers")



In [ ]:
# df_gam = pd.read_csv("../data/train_outliers_rows_GAM.csv")
# df_rf  = pd.read_csv("../data/train_outliers_rows_RF.csv")
# df_brt = pd.read_csv("../data/train_outliers_rows_BRT.csv")
# df_mlp = pd.read_csv("../data/train_outliers_rows_MLP.csv")

# df_gam['Model'] = 'GAM'
# df_rf['Model']  = 'RF'
# df_brt['Model'] = 'BRT'
# df_mlp['Model'] = 'MLP'

# # Combine into one DataFrame
# combined_df = pd.concat([df_gam, df_rf, df_brt, df_mlp], ignore_index=True)

# # Save all
# combined_df.to_csv("../data/train_all_outliers.csv")



#### 3.1 Common outlers of EDM

In [ ]:
# Convert to integer type in order to handle NaNs
outliers_df = outliers_df.astype("Int64")

# Create sets of outlier indices per model 
model_sets = {col: set(outliers_df [col].dropna().astype(int)) for col in outliers_df.columns}

# Common indices across all models 
common_all = set.intersection(*model_sets.values())

# Find pairwise and multi-model intersections
for r in range(2, len(model_sets) + 1):
    for combo in combinations(model_sets.keys(), r):
        inter = set.intersection(*(model_sets[m] for m in combo))
        if inter:
            common_subsets[combo] = inter

print("Common Outliers Across ALL Models:")
print(sorted(common_all))
print()

print("Overlap (counts):")
for combo, inter in sorted(common_subsets.items(), key=lambda x: (len(x[0]), x[0])):
    print(f"{combo}: {len(inter)}")
print()

print("Overlaps (Indices):")
for combo, inter in sorted(common_subsets.items(), key=lambda x: (len(x[0]), x[0])):
    print(f"{combo}: {sorted(inter)}")

In [ ]:
# Mark the three represenatative examples on the box plot of EDM or SHAP discrepancy
records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[avg_error_ref_ranked_df['Reference_Model'] == model].copy()
    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df,
    x='Reference_Model',
    y='Mean_over_SDofMean',
    hue='Reference_Model',
    order=model_order,
    palette=ref_palette,
    showfliers=False
)

# mark only the three representative examples
sites_to_mark = {200, 158, 1419}

jitter_strength = 0.15
np.random.seed(123)

for i, model in enumerate(model_order):
    model_df = df[df['Reference_Model'] == model]

   
    x_vals = i + np.random.uniform(-jitter_strength, jitter_strength, size=len(model_df))
    y_vals = model_df['Mean_over_SDofMean'].values
    sites = model_df['Site'].values

    # keep only the three representative examples
    mask = np.isin(sites, list(sites_to_mark))
    x_vals_selected = x_vals[mask]
    y_vals_selected = y_vals[mask]
    sites_selected = sites[mask]

   
    base_color = to_rgb(ref_palette[model])
    dot_color = tuple([0.8 * c for c in base_color])

    # plot only the three representative examples
    plt.scatter(
        x_vals_selected,
        y_vals_selected,
        color=dot_color,
        alpha=0.6,
        s=30,
        zorder=2
    )

    # labels in red (same as your original)
    for xi, yi, site in zip(x_vals_selected, y_vals_selected, sites_selected):
        plt.text(
            xi + 0.04,
            yi,
            str(site),
            fontsize=10,
            color='red',
            rotation=0,
            ha='left',
            va='center',
            zorder=3
        )

plt.xlabel("Reference Model", fontsize=16, labelpad=10)
# plt.ylabel("Mean / SD(Mean)", fontsize=14)
# plt.ylabel(r"$\mu(i)\,/ \,\sigma_{\mu}$", fontsize=16, labelpad=10)
# plt.title("Training", fontsize=16)
# plt.ylabel(r"$\mu_{j_{1}} (i)/\sigma_{\mu_{j_{1}}}$", fontsize=16, labelpad=10)
plt.ylabel("SHAP discrepancy", fontsize=16, labelpad=10)
plt.tick_params(axis='x', labelsize=14)
plt.tick_params(axis='y', labelsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
model_order = ['GAM','RF', 'BRT', 'MLP']

records = []
for model in model_order:
    sub_df = avg_error_ref_ranked_df[
        avg_error_ref_ranked_df['Reference_Model'] == model
    ].copy()

    sd = sub_df['Average_Relative_Error'].std(ddof=1)
    sub_df['Mean_over_SDofMean'] = (
        sub_df['Average_Relative_Error'] / sd if sd != 0 else np.nan
    )

    records.append(sub_df)

df = pd.concat(records).dropna(subset=['Mean_over_SDofMean'])

#  Single Boxplot
plt.figure(figsize=(3, 6))

sns.boxplot(
    y=df['Mean_over_SDofMean'],
    color='lightgrey',
    showfliers=False,
    width=0.4
)

# Mark only selected sites for MLP 
sites_to_mark = {200, 158, 1419}

# custom labels
site_label_map = {
    200: r"$E_{3}$",
    158: r"$E_{2}$",
    1419: r"$E_{1}$"
}

# Filter only MLP
mlp_df = df[df['Reference_Model'] == 'MLP']

jitter_strength = 0.08
np.random.seed(125)

x_center = 0  # only one box

x_vals = x_center + np.random.uniform(
    -jitter_strength,
    jitter_strength,
    size=len(mlp_df)
)

y_vals = mlp_df['Mean_over_SDofMean'].values
sites = mlp_df['Site'].values

mask = np.isin(sites, list(sites_to_mark))

x_selected = x_vals[mask]
y_selected = y_vals[mask]
sites_selected = sites[mask]

# Plot selected MLP points
plt.scatter(
    x_selected,
    y_selected,
    color='black',
    s=60,
    zorder=3
)

# Red labels
for xi, yi, site in zip(x_selected, y_selected, sites_selected):
    plt.text(
        xi + 0.02,
        yi,
        site_label_map[site],
        fontsize=11,
        fontweight='bold',
        color='red',
        rotation=0,
        ha='left',
        va='center',
        zorder=4
    )

#  Formatting 
plt.xticks([])
plt.ylabel("SHAP discrepancy", fontsize=16, labelpad=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()